# Mortgage complaint operations

**Daniel Umeadi — SQL + Python**

Which complaint issues dominate workload, and where should response timeliness be reviewed?

CFPB Mortgage CSV export retrieved 2026-10-10; immutable bundled snapshot with SHA-256. Analysis locally enforces 2024-01-01 ≤ receipt < 2025-01-01 because returned max dates differ from API documentation.

## 1. Rebuild the analysis

Run the audited pipeline from this notebook directory. Source data is downloaded only when absent; hashes are checked before use.

In [1]:
import sys, subprocess
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
completed = subprocess.run([sys.executable, 'run.py'], check=True, capture_output=True, text=True)
print(completed.stdout)


{
  "complaints": 21438,
  "known_timeliness": 21438,
  "late_count": 240,
  "late_rate": 0.01119507416736636,
  "same_day_forwarding_share": 0.8452280996361601,
  "top_two_issue_workload_share": 0.7876201138165874,
  "largest_issue": "Trouble during payment process",
  "largest_issue_count": 11743,
  "peak_month": "2024-02",
  "peak_month_count": 1942
}



## 2. Inspect the result and audit

Read denominators and exclusions before interpreting the findings.

In [2]:
summary=json.loads(Path('outputs/summary.json').read_text())
display(pd.Series(summary,name='value').to_frame())
checks=json.loads(Path('outputs/quality_checks.json').read_text())
display(pd.Series(checks,name='value').to_frame())

,value
complaints,21438
known_timeliness,21438
late_count,240
late_rate,0.011195
same_day_forwarding_share,0.845228
top_two_issue_workload_share,0.78762
largest_issue,Trouble during payment process
largest_issue_count,11743
peak_month,2024-02
peak_month_count,1942


,value
download_rows,21473
analysis_rows,21438
excluded_outside_2024_or_product,35
invalid_received_dates,0
missing_complaint_ids,0
duplicate_complaint_ids,0
negative_forwarding_intervals,0
unknown_timeliness,0
sql_pandas_count_reconcile,True
monthly_total_reconcile,True


## 3. Inspect the analytical output

In [4]:
display(pd.read_csv('outputs/issue_priority.csv'))


,issue,complaints,known_timeliness,late_count,late_rate,median_forwarding_days,workload_share,cumulative_workload_share,wilson_low,wilson_high
0,Trouble during payment process,11743,11743,95.0,0.008090,0.0,0.547766,0.547766,6.622870e-03,0.009879
1,Struggling to pay mortgage,5142,5142,34.0,0.006612,0.0,0.239854,0.787620,4.735701e-03,0.009225
2,Applying for a mortgage or refinancing an exis...,2154,2154,63.0,0.029248,0.0,0.100476,0.888096,2.292712e-02,0.037245
3,Closing on a mortgage,1427,1427,41.0,0.028732,0.0,0.066564,0.954660,2.124905e-02,0.038745
4,Incorrect information on your report,531,531,4.0,0.007533,0.0,0.024769,0.979429,2.933159e-03,0.019207
5,Problem with a company's investigation into an...,297,297,2.0,0.006734,0.0,0.013854,0.993283,1.848616e-03,0.024217
6,Improper use of your report,97,97,1.0,0.010309,0.0,0.004525,0.997808,1.822118e-03,0.056106
7,Credit monitoring or identity theft protection...,30,30,0.0,0.000000,0.0,0.001399,0.999207,0.000000e+00,0.113517
8,Unable to get your credit report or credit score,11,11,0.0,0.000000,0.0,0.000513,0.999720,2.775558e-17,0.258840
9,Problem with fraud alerts or security freezes,6,6,0.0,0.000000,0.0,0.000280,1.000000,0.000000e+00,0.390343


## 4. Interpretation

21,438 mortgage complaints in the 2024 receipt cohort; the two largest issues represent 78.76%. There are 240 late labels (1.12% of known labels).

**Limits:** Complaints are selected, not random or market-share-adjusted. They are not verified wrongdoing. Timely response is not resolution. Confidence intervals assume independent binary outcomes and do not account for clustering. No narratives or personal-profile modeling is performed.

**Next decision:** Which complaint issues dominate workload, and where should response timeliness be reviewed? 

## Charts

![01_monthly](outputs/01_monthly.png)

![02_issues](outputs/02_issues.png)

![03_timeliness](outputs/03_timeliness.png)

![04_forwarding](outputs/04_forwarding.png)

